# Setting up API
Source: NASS



In [25]:
from dotenv import load_dotenv
import os
import pandas as pd
import requests
from pathlib import Path

In [26]:
load_dotenv()
API_KEY = os.getenv("NASS_API_KEY")

base_url = 'https://quickstats.nass.usda.gov/api/api_GET/'

dfs = []

# Yield API call
for crop in ['CORN', 'WHEAT']:
    params = {
            'key' : API_KEY,
            'source_desc' : 'SURVEY',
            'sector_desc' : 'CROPS',
            'group_desc' : 'FIELD CROPS',
            'commodity_desc': crop,
            'agg_level_desc' : 'STATE',
            'freq_desc' : 'ANNUAL',
            'statisticcat_desc' : 'YIELD',
            'prodn_practice_desc': 'ALL PRODUCTION PRACTICES',
            'year__GE': '1961',
            'format' : 'JSON'}
    # Explicit class rule for Wheat
    if crop == "WHEAT":
        params["class_desc"] = "ALL CLASSES"

    r = requests.get(base_url, params=params)
    print(r.status_code)
    print(r.text[:1000])
    r.raise_for_status()

    data = r.json()["data"]
    df = pd.DataFrame(data)
    df["commodity"] = crop

    dfs.append(df)

df_yield = pd.concat(dfs, ignore_index=True)

200
{"data":[{"class_desc":"ALL CLASSES","source_desc":"SURVEY","prodn_practice_desc":"ALL PRODUCTION PRACTICES","location_desc":"OTHER STATES","region_desc":"","congr_district_code":"","freq_desc":"ANNUAL","util_practice_desc":"GRAIN","end_code":"00","group_desc":"FIELD CROPS","asd_code":"","reference_period_desc":"YEAR","domain_desc":"TOTAL","CV (%)":"","year":2025,"country_code":"9000","begin_code":"00","agg_level_desc":"STATE","sector_desc":"CROPS","week_ending":"","county_name":"","short_desc":"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE","watershed_desc":"","county_code":"","asd_desc":"","unit_desc":"BU / ACRE","state_alpha":"OT","commodity_desc":"CORN","state_name":"OTHER STATES","watershed_code":"00000000","zip_5":"","statisticcat_desc":"YIELD","country_name":"UNITED STATES","Value":"0","state_fips_code":"98","state_ansi":"","load_time":"2026-01-12 12:00:00.000","domaincat_desc":"NOT SPECIFIED","county_ansi":""},{"statisticcat_desc":"YIELD","commodity_desc":"CORN","unit_desc":"B

In [27]:
dfs_area = []

# Area Harvested API call
for crop in ['CORN', 'WHEAT']:
    params = {
            'key' : API_KEY,
            'source_desc' : 'SURVEY',
            'sector_desc' : 'CROPS',
            'group_desc' : 'FIELD CROPS',
            'commodity_desc': crop,
            'agg_level_desc' : 'STATE',
            'freq_desc' : 'ANNUAL',
            'statisticcat_desc' : 'AREA HARVESTED',
            'prodn_practice_desc': 'ALL PRODUCTION PRACTICES',
            'year__GE': '1961',
            'format' : 'JSON'}
    # Explicit class rule for Wheat
    if crop == "WHEAT":
        params["class_desc"] = "ALL CLASSES"

    r = requests.get(base_url, params=params)
    print(r.status_code)
    print(r.text[:1000])
    r.raise_for_status()

    data = r.json()["data"]
    df = pd.DataFrame(data)
    df["commodity"] = crop

    dfs_area.append(df)

df_area = pd.concat(dfs_area, ignore_index=True)

200
{"data":[{"sector_desc":"CROPS","agg_level_desc":"STATE","week_ending":"","short_desc":"CORN, FORAGE - ACRES HARVESTED","county_name":"","county_code":"","watershed_desc":"","year":1985,"begin_code":"00","country_code":"9000","Value":"40,000","country_name":"UNITED STATES","state_ansi":"38","state_fips_code":"38","load_time":"2012-01-01 00:00:00.000","domaincat_desc":"NOT SPECIFIED","county_ansi":"","commodity_desc":"CORN","unit_desc":"ACRES","state_alpha":"ND","asd_desc":"","zip_5":"","state_name":"NORTH DAKOTA","watershed_code":"00000000","statisticcat_desc":"AREA HARVESTED","location_desc":"NORTH DAKOTA","region_desc":"","freq_desc":"ANNUAL","congr_district_code":"","util_practice_desc":"FORAGE","class_desc":"ALL CLASSES","source_desc":"SURVEY","prodn_practice_desc":"ALL PRODUCTION PRACTICES","end_code":"00","asd_code":"","group_desc":"FIELD CROPS","domain_desc":"TOTAL","reference_period_desc":"YEAR","CV (%)":""},{"week_ending":"","sector_desc":"CROPS","agg_level_desc":"STATE","

# Cleaning Yield Dataframe

In [28]:
df_y = df_yield.copy()

# Keeping BU/ACRE
df_y = df_y[df_y["unit_desc"] == "BU / ACRE"].copy()

# Keeping ALL util practice
df_y = df_y[df_y["util_practice_desc"] == "ALL UTILIZATION PRACTICES"].copy()

# Dropping state aggregates
df_y = df_y[df_y["state_alpha"] != "OT"].copy()

# Col list we keep
COLS_YIELD = ["state_alpha", "state_name", "year",
             "commodity_desc", "unit_desc", "Value",
             "reference_period_desc", 'source_desc']

# Filtering out unnecessary cols
df_y = df_y[COLS_YIELD].rename(columns={
    "commodity_desc": "crop",
    "unit_desc": "yield_unit",
    "Value": "yield"
})

# Picking max value from yearly forecasts and year
df_y = (df_y.sort_values(["state_alpha","year","crop","yield"])   # YEAR=0 will be smaller than NOV>0
          .groupby(["state_alpha","year","crop"], as_index=False)
          .tail(1)
          .reset_index(drop=True))

print(df_y.duplicated(["state_alpha","year","crop"]).sum())

0


# Cleaning Area Harvested Dataframe

In [29]:
df_a = df_area.copy()

# Keeping ACRES
df_a = df_a[df_a[('unit_desc')] == "ACRES"].copy()

# Keeping all util practices
df_a = df_a[df_a["util_practice_desc"] == "ALL UTILIZATION PRACTICES"].copy()

# Keeping year only
df_a = df_a[df_a["reference_period_desc"] == "YEAR"].copy()

# Dropping OT
df_a = df_a[df_a["state_alpha"] != "OT"].copy()

# Filtering out unnecessary columns
COLS_AREA = ["state_alpha", "state_name", "year",
             "commodity_desc", "unit_desc", "Value",
             "reference_period_desc", 'source_desc']

df_a = df_a[COLS_AREA].rename(columns={
    "commodity_desc": "crop",
    "unit_desc": "area_unit",
    "Value": "area_harvested"
})

# Picking max value from yearly forecasts and year
df_a = (df_a.sort_values(["state_alpha", "year", "crop", 'area_harvested'])
          .groupby(["state_alpha", "year", "crop"], as_index=False)
          .tail(1)
          .reset_index(drop=True))

df_a.duplicated(["state_alpha", "year", "crop"]).sum()


np.int64(0)

# Merging Dataframes

In [30]:
merged = df_y[["state_alpha", "year", "crop","yield"]].merge(
    df_a[["state_alpha", "year", "crop","area_harvested"]],
    on=["state_alpha", "year", "crop"],
    how="inner"
)

display(merged.head())

,state_alpha,year,crop,yield,area_harvested
0,AL,1961,WHEAT,26,"56,000"
1,AL,1962,WHEAT,24,"35,000"
2,AL,1963,WHEAT,23.5,"42,000"
3,AL,1964,WHEAT,25,"64,000"
4,AL,1965,WHEAT,24.5,"55,000"


# Saving Dataset

In [31]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
merged.to_parquet("../data/processed/crop_annual.parquet", index=False)